# A1 · Reducción raw (esorex)

**Spec:** [`docs/spec_A1_codex_raw_reduction.md`](../docs/spec_A1_codex_raw_reduction.md)  |  **Bloque:** A · Reducción  |  **Run de este set:** `ROXs12b_realigned`

Reduce los raw MUSE con esorex y alinea las exposiciones hasta `cube_telcorr.fits`.

| | |
|---|---|
| **Entrada** | Raw MUSE + calibraciones |
| **Salida (QC/productos)** | `cube_telcorr.fits`, `stages/stage00r_qc.json` |
| **Consume aguas abajo** | Todo el bloque B |


## Cómo ejecutar de forma independiente

Etapa de **reducción**: la celda de abajo resuelve el comando real para **este objeto** a partir de su `chain.reduction_profile` y de su config, y puede lanzarlo. Son trabajos largos (ver coste), así que se lanzan en segundo plano con el log a la vista; el notebook no se bloquea.

Si algún dato no está declarado en el config del run, la celda lo dice y **no lanza** en vez de inventarse una ruta.

Comando histórico de referencia:

```bash
conda activate MUSE
bash scripts/reduce_raw.sh
```


## Coste de ejecución (esorex)

> ⏱️ **Referencia real** medida en esta máquina (esorex 3.13.10 / MUSE 2.10.16, dataset NFM-AO de ROXs 12: **7 exposiciones × 24 IFUs = 168 pixtables**).

| Receta | Tiempo | Escala con |
|---|---:|---|
| bias | 33 min | calibración (~fijo) |
| flat | 52 min | calibración |
| wavecal | 51 min | calibración |
| lsf | 50 min | calibración |
| scibasic (std) | 4.5 min | 1× |
| standard | 1.6 min | 1× |
| scibasic (object) | 24 min | **N_exp** |
| scipost | 73 min | **N_exp** |
| **Total (7 exp)** | **≈ 289 min (~4.8 h)** | |

**Fórmula para datos nuevos** (mismo instrumento/máquina):

```
T(min) ≈ T_cal + T_std + N_exp·(t_scibasic + t_scipost) + T_combine
```

con constantes medidas aquí:

- `T_cal ≈ 186 min` = bias+flat+wavecal+lsf. **Una vez por noche/modo**; `0` si reutilizas los master calibrations.
- `T_std ≈ 6 min` = scibasic_std + standard (una vez).
- `t_scibasic ≈ 3.4 min/exp`, `t_scipost ≈ 10.5 min/exp` (24 IFUs c/u; plan B = scipost por exposición).
- `T_combine ≈ 5 min` = muse_exp_align + muse_exp_combine (plan B, offsets manuales).

**Nota:** scibasic/scipost paralelizan sobre los 24 IFUs (OpenMP) → el tiempo escala aprox. inverso al nº de núcleos; `cores_factor` ajusta ese factor respecto a esta máquina base (=1.0). La calibración domina: reutilizar masters recorta ~3 h.


In [ ]:
def estimate_esorex_runtime(n_exp, reuse_calibrations=False, cores_factor=1.0):
    """Estima el wall-time de la reducción esorex (min), calibrada en la
    máquina de referencia (7 exp NFM-AO ~= 289 min). Ver tabla de arriba."""
    T_cal = 0.0 if reuse_calibrations else 186.0  # bias+flat+wavecal+lsf
    T_std = 6.0                                    # scibasic_std + standard
    t_scibasic, t_scipost = 3.4, 10.5             # min por exposición (24 IFU)
    T_combine = 5.0                                # exp_align + exp_combine
    return (T_cal + T_std + n_exp * (t_scibasic + t_scipost) + T_combine) / cores_factor

for n in (1, 3, 7, 10):
    m = estimate_esorex_runtime(n)
    print(f'{n:2d} exp  ->  {m:5.0f} min  (~{m/60:.1f} h)')
print('7 exp reutilizando masters ->',
      f'{estimate_esorex_runtime(7, reuse_calibrations=True):.0f} min')


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage00r_qc.json', RUN_ID))


## Mapa de la cadena de este objeto

Qué etapas están ejecutadas, en **qué run** vive el QC de cada una y con qué fecha. El reparto entre runs se declara en `chain` dentro de `runs/<run>/config/config.json` (clave `stage_runs`); una etapa marcada `no ejecutada` no es un error, es trabajo pendiente para este objeto. Un `!` (CROSS-OBJECT) sí es un problema: se estaría leyendo otro objeto.

Ver `docs/plan_multiobjeto_notebooks_2026-07-24.md`.


In [ ]:
nb.show_chain(RUN_ID)


## Ejecutar o auditar


In [ ]:
cmd, target_run, missing = nb.launch_command('A1', RUN_ID)
print('run que ejecuta esta etapa:', target_run)
print('comando resuelto para este objeto:')
print('   ', cmd or '(sin plantilla)')
if missing:
    print()
    print('NO se puede lanzar: faltan datos en el config del run.')
    print('   sin resolver:', ', '.join(missing))
    print(f'   declara esas claves en runs/{target_run}/config/config.json')

RUN = False   # -> True para LANZAR (trabajo largo: revisa el coste arriba)

if RUN and not missing:
    import subprocess, time
    from pathlib import Path
    log = Path(nb.run_dir(target_run)) / 'logs' / f'a1_launch.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, 'w') as fh:
        proc = subprocess.Popen(cmd, shell=True, cwd=str(nb.project_root()),
                                stdout=fh, stderr=subprocess.STDOUT)
    print(f'lanzado en segundo plano (pid {proc.pid}); log -> {log}')
    print('sigue el progreso con:  !tail -f', log)
elif RUN:
    print('RUN=True pero hay datos sin resolver: no se lanza nada.')
else:
    print()
    print('Modo auditoría (RUN=False): abajo se carga el QC existente.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage00r_qc.json', RUN_ID)
nb.show(qc, keys=['shape', 'sha', 'offset', 'esorex', 'muse'], title='A1')


## Verificaciones (V1–V6): qué comprueban y qué respondieron

El QC de A1 (`stage00r_qc.json`) no re-reduce: **verifica** que el cubo entregado es sano y trazable. Cada check tiene un significado concreto:

| Check | Qué comprueba | Resultado | Significado |
|---|---|---|---|
| **V1** STAT | La extensión STAT (varianza) existe, es positiva y con pocos NaN (excluyendo canales láser AO y spaxels de borde) | **ok** (NaN 0.24%, 216 canales láser y 6972 spaxels de borde excluidos) | El cubo trae su mapa de varianza y no está corrupto → base para toda la propagación de error aguas abajo |
| **V2** estándar | Continuo del estándar vs su curva de respuesta | **unavailable** | No hay curva de respuesta del estándar para esta reducción → no se pudo cerrar la validación de flujo relativa aquí (se cierra por otra vía en A4/M3 vs Gaia) |
| **V3** WCS | `CRVAL3` y paso espectral correctos | **ok** | Solución de longitud de onda y WCS sanos → los λ del cubo son fiables |
| **V4** vs ADP | Correlación de la imagen luz-blanca del cubo propio con la del ADP de ESO | **ok** (corr = 0.9994, shift entero (0,0)) | El cubo auto-reducido reproduce la morfología del ADP oficial → validación cruzada independiente de la reducción |
| **V5** espectro estelar | Razón del espectro de la estrella entre cubo y ADP | **unavailable** | Los dos cubos ponen la estrella en píxeles distintos; hace falta registro por-cubo → no comparable con la API punto-único |
| **V6** máscara de cielo | La máscara de cielo de scipost es limpia | **unavailable** | scipost no exporta la máscara como producto 2D verificable → no auditable aquí |

V1/V3/V4 pasan; V2/V5/V6 son **lagunas de proveniencia documentadas** (no fallos físicos). Por eso el semáforo A1 = **yellow**. La celda de abajo los imprime en vivo.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('A1', 'stages/stage00r_qc.json'):
        q = nb.load_qc('stages/stage00r_qc.json', RUN_ID)
        profile = nb.chain_of(RUN_ID).get('reduction_profile', '(no declarado)')
        labels = {
            'v1_stat_present':       'V1 · STAT presente y sano',
            'v2_std_residual':       'V2 · Residuo del estándar (respuesta de flujo)',
            'v3_wcs_ok':             'V3 · WCS / eje espectral',
            'v4_adp_whitelight':     'V4 · Correlación luz-blanca vs ADP',
            'v5_adp_star_spec_ratio':'V5 · Razón de espectro estelar vs ADP',
            'v6_sky_mask_clean':     'V6 · Máscara de cielo limpia',
        }
        ver = q.get('verification', {})
        print(f'perfil de reducción: {profile}   ({len(ver)} verificaciones en el QC)')
        print()
        for prefix, lab in labels.items():
            # los nombres difieren por sufijo entre variantes (p.ej. _rms)
            key = next((k for k in ver if k.startswith(prefix)), None)
            if key is None:
                print(f'{lab}\n   -> ausente en esta variante de QC\n')
                continue
            v = ver[key]
            if isinstance(v, dict):
                res = 'ok' if v.get('ok') else v.get('status', '?')
                msg = v.get('message', '')
            elif v is None:
                res, msg = 'unavailable', 'sin medir en esta reducción'
            else:
                res, msg = ('ok' if v else 'no'), ''
            print(f'{lab}\n   -> {res}\n   {msg}\n')


## Decisiones y notas
- **Alineación por plan B (OFFSET_LIST manual)**, no `exp_align` — daba offsets espurios de hasta 3.305" (cross-match de speckles NFM); el manual desde el centroide de la primaria da máx 0.62". El cubo realineado ≡ ADP a través del bloque B.
- Provenance QC = **AMARILLO**: V1/V3/V4 pasan; V2/V5/V6 = `unavailable` (lagunas documentadas, no fallos). Ver tabla de verificaciones arriba.
- **Estado A-block (actualizado 2026-07-19):** los 6 blockers duros están **CERRADOS** → F1 realineado = `yellow`, **0 bloqueantes**. Los `open_issues` restantes (V2/V5/V6) quedan documentados. **Track A cerrado:** agrupación BIAS aceptada (A2b, impacto negligible) y telúrica justificada (A1b) + **molecfit converge y corrobora STD_TELLURIC (A1a, 2026-07-19)**. El paquete ya **no bloquea por A**; la validez para paper es juicio científico con esos caveats declarados. **Supera la directiva absoluta del 2026-07-07.**


## Checks


In [ ]:
print('open_issues A1 (no bloqueantes):')
for i, s in enumerate(qc.get('open_issues', []), 1):
    print(f'  {i}. {s}')


## Conclusión (registrada)

**A1: `cube_telcorr.fits` reducido y alineado; semáforo A1 = `yellow` (no bloqueante).**

- **Fecha:** reducción 2026-07-08 (esorex 3.13.10 / MUSE 2.10.16); provenance QC escrito 2026-07-09.
- **Datos:** 7 exposiciones NFM-AO (OB 3444577, Prog 109.23B7.002, MUSE.2022-09-01T00:36–02:03).
- **Alineación:** plan B, OFFSET_LIST manual desde el centroide de la primaria (máx 0.62"), porque `muse_exp_align` dio offsets espurios de hasta 3.305".
- **Verificaciones:** V1/V3/V4 pass; V2/V5/V6 `unavailable` (documentadas).
- **Cubo:** 3681×330×338, sha256 `9fff16b7…`; corr luz-blanca vs ADP = 0.9994.
- **A-block:** 6 blockers duros cerrados (F1 yellow, 0 bloqueantes); 4 caveats no bloqueantes documentados. Nada es aún paper-final sin declarar esos caveats.
